In [2]:
import requests
import json
import time
import os
from tqdm import tqdm
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

id_to_question = {}
conditionId_to_id = {}
all_data = []

start_id = 405000
end_id = 505566
delay_between_requests = 0.1
batch_size = 500
batch_number = 1

session = requests.Session()
retry_strategy = Retry(
    total=5,  # Total number of retries
    status_forcelist=[429, 500, 502, 503, 504],  # Retry on these HTTP status codes
    method_whitelist=["GET"],  # Retry only on GET requests
    backoff_factor=1  # Exponential backoff factor (e.g., 1, 2, 4, 8, ...)
)

adapter = HTTPAdapter(max_retries=retry_strategy)
session.mount("https://", adapter)
session.mount("http://", adapter)

os.makedirs('market_data', exist_ok=True)


for index, market_id in enumerate(tqdm(range(start_id, end_id + 1)), start=1):
    url = f'https://gamma-api.polymarket.com/markets/{market_id}'
    try:
        response = session.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if 'error' not in data:
                market_id_str = data.get('id', '')
                question = data.get('question', '')
                condition_id = data.get('conditionId', '')
                
                id_to_question[market_id_str] = question
                conditionId_to_id[condition_id] = market_id_str
                all_data.append(data)
        else:
            print(f"Unexpected status code {response.status_code} for market ID {market_id}")

    except Exception as e:
        print(f"Exception occurred for market ID {market_id}: {e}")

    finally:
        time.sleep(delay_between_requests)

    # Check if the batch should be saved and if files for this batch already exist
    if index % batch_size == 0:
        data_file = f'market_data/all_market_data_batch_{batch_number}.json'

        # Only save if files don't already exist
        if not (os.path.exists(data_file)):
            with open(data_file, 'w') as f:
                json.dump(all_data, f, indent=2)

        # Clear data for the next batch
        id_to_question.clear()
        conditionId_to_id.clear()
        all_data.clear()
        batch_number += 1

# Save any remaining data if not saved in previous batch
if all_data:
    data_file = f'market_data/all_market_data_batch_{batch_number}.json'
    
    if not os.path.exists(data_file):
        try:
            with open(data_file, 'w') as f:
                json.dump(all_data, f, indent=2)
            print(f"Final batch {batch_number} saved with {len(all_data)} records.")
        except Exception as e:
            print(f"Error saving final batch {batch_number}: {e}")

print("Data fetching complete. Mappings and data have been saved to JSON files.")

TypeError: Retry.__init__() got an unexpected keyword argument 'method_whitelist'